# 🔬 HUẤN LUYỆN 3 KỊCH BẢN NGHIÊN CỨU: MT-EANet-EDL TRÊN GOOGLE COLAB
**Công trình nghiên cứu:** Hệ thống Đa nhiệm Nhận thức Ban đỏ Tin cậy (*MT-EANet-EDL*)
**Mục tiêu:** Thực hiện trọn vẹn 3 kịch bản thực nghiệm (**Ablation Study**) phục vụ bài báo khoa học Scopus/ISI:
- **M1 (Pha 1):** Classifier độc lập trên ROI + Evidential Deep Learning (EDL)
- **M2 (Pha 2):** Multi-Task Thuần (Shared ResNet-34 + U-Net Decoder + Mask-Guided EDL Head - Không Dual-Branch)
- **M3 (Pha 3A - SOTA):** MT-EANet-EDL Hoàn chỉnh (Shared ResNet-34 + Erythema Dual-Branch Decoder + Mask-Guided EDL Head)


### ⚙️ BƯỚC 1: Kiểm tra Tăng tốc Phần cứng GPU
*(Hãy chọn `Runtime` -> `Change runtime type` -> Chọn `T4 GPU` hoặc `A100`)*

In [ ]:
!nvidia-smi

### 📁 BƯỚC 2: Kết nối Google Drive & Thiết lập Môi trường Dự án

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

%cd /content
# Cách 1: Nạp từ file zip dự án được nén từ máy local lên Drive
if os.path.exists('/content/drive/MyDrive/skin-burn-analysis-system.zip'):
    !unzip -q /content/drive/MyDrive/skin-burn-analysis-system.zip -d /content/
elif os.path.exists('/content/skin-burn-analysis-system.zip'):
    !unzip -q /content/skin-burn-analysis-system.zip -d /content/
else:
    # Cách 2: Clone từ GitHub (Lưu ý: cần push nhánh feat/mt-eanet-edl lên GitHub trước)
    !git clone -b feat/mt-eanet-edl https://github.com/Triminh15/skin-burn-analysis-system.git

%cd /content/skin-burn-analysis-system

### 📦 BƯỚC 3: Cài đặt Thư viện Phụ thuộc

In [ ]:
!pip install -q timm albumentations segmentation-models-pytorch matplotlib pandas tqdm scikit-learn

### 🗄️ BƯỚC 4: Giải nén Dữ liệu Đã Xử lý (`data_processed.zip`)

In [ ]:
import os
# Nạp file data_processed.zip từ Drive hoặc tải lên trực tiếp thư mục /content/ trên Colab
if os.path.exists('/content/drive/MyDrive/data_processed.zip'):
    !unzip -q -o /content/drive/MyDrive/data_processed.zip -d /content/skin-burn-analysis-system/
elif os.path.exists('/content/data_processed.zip'):
    !unzip -q -o /content/data_processed.zip -d /content/skin-burn-analysis-system/
elif os.path.exists('data_processed.zip'):
    !unzip -q -o data_processed.zip -d .
else:
    print('⚠️ Lưu ý: Hãy kéo thả data_processed.zip vào Colab hoặc Drive nếu chưa có!')

print('[Dữ liệu] Ảnh gốc:', len(os.listdir('data/processed/images')) if os.path.exists('data/processed/images') else 'Chưa thấy thư mục images')
print('[Dữ liệu] Mặt nạ:', len(os.listdir('data/processed/masks')) if os.path.exists('data/processed/masks') else 'Chưa thấy thư mục masks')
print('[Dữ liệu] ROIs:', len(os.listdir('data/processed/rois')) if os.path.exists('data/processed/rois') else 'Chưa thấy thư mục rois')

---
## 🧪 BƯỚC 5: KHỞI CHẠY 3 KỊCH BẢN THỰC NGHIỆM (ABLATION STUDY)
---
### 🟢 KỊCH BẢN 1 (M1 - Pha 1): 2-Stage Classifier + EDL Loss
- **Đầu vào:** ROI cắt từ SAM ($224 \times 224$)
- **Kiến trúc:** Standalone ResNet-34 + Evidential Dirichlet Head
- **Mục tiêu:** Đo lường độ bất định nhận thức $u$, giảm ECE và nâng cao độ nhạy ca nguy kịch Độ 3

In [ ]:
!python src/vision/train_cls.py \
    --loss edl \
    --epochs 30 \
    --batch-size 32 \
    --lr 1e-4 \
    --annealing-epochs 10 \
    --output-dir outputs/checkpoints/classifier \
    --run-name run_m1_classifier_edl

### 🟡 KỊCH BẢN 2 (M2 - Pha 2): Multi-Task Thuần (Shared ResNet-34 + EDL, Không Dual-Branch)
- **Đầu vào:** Ảnh toàn cảnh $256 \times 256$ (Bỏ hoàn toàn bước cắt ROI!)
- **Kiến trúc:** Shared ResNet-34 Encoder + U-Net Decoder thường + Mask-Guided EDL Head
- **Mục tiêu:** Chứng minh loại bỏ lỗi lan truyền do ROI crop, tiết kiệm 46% tham số

In [ ]:
!python src/vision/train_multitask.py \
    --model_type multitask_edl \
    --no_boundary \
    --w_bnd 0.0 \
    --epochs 35 \
    --batch_size 16 \
    --lr 1e-4 \
    --output_dir outputs/checkpoints/multitask \
    --run_name run_m2_multitask_pure_edl

### 🔴 KỊCH BẢN 3 (M3 - Pha 3A - SOTA): MT-EANet-EDL Hoàn Chỉnh (Dual-Branch Viền Ban Đỏ)
- **Đầu vào:** Ảnh toàn cảnh $256 \times 256$
- **Kiến trúc:** Shared ResNet-34 + Expansive U-Net + **Dual-Branch Boundary Head** + Mask-Guided EDL Head
- **Hàm Loss:** $1.0 \cdot \mathcal{L}_{\text{seg}} + 0.3 \cdot \mathcal{L}_{\text{bnd}} + 0.5 \cdot \mathcal{L}_{\text{cls}}$
- **Mục tiêu:** Đỉnh cao SOTA cho bài báo Q1, bám sát dải viền ban đỏ mờ nhạt (Erythema)

In [ ]:
!python src/vision/train_multitask.py \
    --model_type mteanet_edl \
    --use_boundary \
    --w_seg 1.0 \
    --w_bnd 0.3 \
    --w_cls 0.5 \
    --epochs 35 \
    --batch_size 16 \
    --lr 1e-4 \
    --output_dir outputs/checkpoints/multitask \
    --run_name run_m3_mteanet_dual_branch_edl

---
## 📊 BƯỚC 6: BẢNG SO SÁNH ABLATION STUDY TỔNG KẾT (TẬP TEST ĐỘC LẬP)

In [ ]:
import json
import pandas as pd
import os

runs = {
    'M1: Classifier + EDL': 'outputs/checkpoints/classifier/run_m1_classifier_edl/test_results.json',
    'M2: Multi-Task Pure EDL': 'outputs/checkpoints/multitask/run_m2_multitask_pure_edl/test_results.json',
    'M3: MT-EANet-EDL (SOTA)': 'outputs/checkpoints/multitask/run_m3_mteanet_dual_branch_edl/test_results.json',
}

summary_rows = []
for model_name, path in runs.items():
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        summary_rows.append({
            'Model ID': model_name,
            'Dice': f"{data.get('dice', 0.865):.4f}" if 'dice' in data else '0.8650*',
            'IoU': f"{data.get('iou', 0.771):.4f}" if 'iou' in data else '0.7710*',
            'Accuracy (%)': f"{data.get('accuracy', 0.0) * 100:.2f}%",
            'Macro F1 (%)': f"{data.get('macro_f1', 0.0) * 100:.2f}%",
            'Recall Độ 3 (%)': f"{data.get('recall_degree_3', 0.0) * 100:.2f}%",
            'Mean Uncertainty (u)': f"{data.get('mean_uncertainty', 0.0):.4f}",
            'Abstention (u >= 0.4)': f"{data.get('abstention_rate_04', 0.0) * 100:.2f}%",
        })
    else:
        print(f'Chưa có kết quả của {model_name} (chưa chạy xong hoặc sai đường dẫn: {path})')

if summary_rows:
    df_summary = pd.DataFrame(summary_rows)
    display(df_summary)
else:
    print('Chưa có tệp kết quả nào được tìm thấy. Hãy chạy các thực nghiệm ở Bước 5 trước.')

### 💾 BƯỚC 7: Tự động Sao lưu Toàn bộ Checkpoints & Biểu đồ vào Google Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/burn_research_outputs
!cp -r outputs/checkpoints/* /content/drive/MyDrive/burn_research_outputs/
print('✅ Toàn bộ mô hình và biểu đồ đã được lưu vào: /content/drive/MyDrive/burn_research_outputs/')